In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import build_dataset
from src.diffusion import DDIMProcess
from src.misc import load_config
from src.predict import load_predict_config, load_predictor


def predict_noise(model, images, steps, condition, guidance, axis_condition):
    if condition is None:
        return model(images, steps, axis_condition=axis_condition)
    if guidance == 1.0:
        return model(images, steps, condition, axis_condition)

    null = model(images, steps, torch.zeros_like(condition), axis_condition)
    guided = model(images, steps, condition, axis_condition)
    return null + guidance * (guided - null)


In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
SAMPLE_COUNT = 4
DDIM_STEPS = 20
AXIS_CONDITION = 0
AXIS_NAMES = ("axis 0", "axis 1", "axis 2")

In [ ]:
config = load_predict_config(PREDICT_CONFIG)
run_dir = ROOT / config.run_dir
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
predictor = load_predictor(run_dir, device=device)
options = config.make_options(predictor)

data_config = Namespace(**load_config(run_dir / "model.yaml"))
data_config.data_dir = {
    axis: (path if (path := Path(value)).is_absolute() else run_dir / path).resolve()
    for axis, value in data_config.data_dir.items()
}
data_config.augment = False
dataset = build_dataset(data_config)
candidate_indices = dataset.condition_indices[AXIS_CONDITION]
indices = torch.randint(len(candidate_indices), (SAMPLE_COUNT,)).tolist()
real = torch.stack(
    [dataset[candidate_indices[index]][0][0] for index in indices]
)

model = predictor.sampler.model
ddpm = predictor.sampler.ddpm
ddim = DDIMProcess(ddpm, min(DDIM_STEPS, ddpm.num_timesteps))
samples = torch.randn(
    SAMPLE_COUNT,
    predictor.num_phases,
    predictor.image_size,
    predictor.image_size,
    device=device,
)
fractions = options.phase_fractions
condition = (
    None
    if fractions is None
    else torch.tensor(fractions, device=device).expand(SAMPLE_COUNT, -1)
)
comparison_steps = torch.full(
    (SAMPLE_COUNT,), ddim.schedule[0][0], dtype=torch.long, device=device
)
with torch.no_grad():
    axis_predictions = [
        predict_noise(
            model,
            samples,
            comparison_steps,
            condition,
            options.guidance_scale,
            torch.full(
                (SAMPLE_COUNT,), axis, dtype=torch.long, device=device
            ),
        )
        for axis in range(3)
    ]
print(
    "same-input mean absolute differences:",
    {
        f"{AXIS_NAMES[left]}-{AXIS_NAMES[right]}": round(
            (axis_predictions[left] - axis_predictions[right])
            .abs()
            .mean()
            .item(),
            6,
        )
        for left, right in ((0, 1), (0, 2), (1, 2))
    },
)
axis_condition = torch.full(
    (SAMPLE_COUNT,), AXIS_CONDITION, dtype=torch.long, device=device
)

with torch.no_grad():
    for step, prev_step in ddim.schedule:
        steps = torch.full((SAMPLE_COUNT,), step, dtype=torch.long, device=device)
        noise = predict_noise(
            model,
            samples,
            steps,
            condition,
            options.guidance_scale,
            axis_condition,
        )
        samples = ddim.step(samples, noise, step=step, prev_step=prev_step)

generated = samples.argmax(dim=1).cpu()
counts = torch.bincount(generated.flatten(), minlength=predictor.num_phases)
sample_fractions = [round(value, 4) for value in (counts / counts.sum()).tolist()]
print(
    f"run={run_dir.name} steps={len(ddim.schedule)} "
    f"fractions={sample_fractions}"
)

In [ ]:
fig, axes = plt.subplots(
    2, SAMPLE_COUNT, figsize=(3 * SAMPLE_COUNT, 8), squeeze=False
)
for row, (images, title) in enumerate(
    (
        (real, f"TRAINING DATA / {AXIS_NAMES[AXIS_CONDITION]}"),
        (generated, f"GENERATED / {AXIS_NAMES[AXIS_CONDITION]}"),
    )
):
    for column, (axis, image) in enumerate(zip(axes[row], images)):
        axis.imshow(
            image,
            cmap="gray",
            vmin=0,
            vmax=predictor.num_phases - 1,
            interpolation="nearest",
        )
        axis.set_title(
            f"{title} {column + 1}", fontsize=16, fontweight="bold", pad=10
        )
        axis.axis("off")
fig.suptitle(
    "Training data vs generated samples", fontsize=22, fontweight="bold", y=0.97
)
fig.subplots_adjust(left=0.03, right=0.99, bottom=0.03, top=0.88, wspace=0.10, hspace=0.35)
plt.show()